In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%run ./_local_config

In [0]:
import pandas as pd
import json
from azure.storage.blob import BlobServiceClient

conn_str = f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};AccountKey={storage_account_key};EndpointSuffix=core.windows.net"

blob_service = BlobServiceClient.from_connection_string(conn_str)
blob_client = blob_service.get_blob_client(container="bronze", blob="erp/battery/battery.json")

stream = blob_client.download_blob().readall()

# Decode bytes to text, split into lines, parse each line as its own JSON object
# (each line here is one full API page response, not one sales record)
text = stream.decode("utf-8-sig")
pages = [json.loads(line) for line in text.splitlines() if line.strip()]

# Each page has a "value" key containing a list of actual sales records —
# flatten all pages into a single list of records
all_records = []
for page in pages:
    all_records.extend(page["value"])

bronze = pd.DataFrame(all_records)

print(bronze.shape)
print(bronze.columns.tolist())
bronze.head()

In [0]:
print(bronze.shape)  # expect something in the hundreds of thousands, matching your earlier ~596K count
print(bronze["itemCategoryCode"].unique())

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.transform.clean_silver import clean_to_silver

silver = clean_to_silver(bronze)
print(silver.shape)
silver.head()

In [0]:
unique_categories = sorted(silver["itemCategoryCode"].dropna().unique())
print(f"Found {len(unique_categories)} unique item categories")
print(unique_categories)

In [0]:
vehicle_counts = silver["vehicle_type"].value_counts(dropna=False)
print(vehicle_counts)

In [0]:
unique_vehicle_types = sorted(silver["vehicle_type"].dropna().unique())

vehicle_type_lookup = pd.DataFrame({
    "vehicle_type_id": [f"V{i}" for i in range(1, len(unique_vehicle_types) + 1)],
    "vehicle_type": unique_vehicle_types
})

print(vehicle_type_lookup)

In [0]:
brand_counts = silver.groupby(["brand_code", "brand_description"]).size().reset_index(name="count")
brand_counts = brand_counts.sort_values("count", ascending=False)
print(brand_counts.to_string())

In [0]:
import json

# Convert DataFrame to JSON records, handling datetime serialization
silver_json = silver.to_json(orient="records", date_format="iso", lines=False)

blob_client = blob_service.get_blob_client(
    container="silver",
    blob="erp/battery/battery_clean.json"
)
blob_client.upload_blob(silver_json, overwrite=True)

print(f"Saved {len(silver)} rows to silver/erp/battery/battery_clean.json")

In [0]:
container_client = blob_service.get_container_client("silver")
for blob in container_client.list_blobs(name_starts_with="erp/battery/"):
    print(blob.name, blob.size)

In [0]:
missing_dates = set(pd.date_range(silver["posting_date"].min(), silver["posting_date"].max())) - set(silver["posting_date"].unique())
missing_df = pd.DataFrame({"missing_date": sorted(missing_dates)})
missing_df["day_of_week"] = pd.to_datetime(missing_df["missing_date"]).dt.day_name()

print(f"Total missing days: {len(missing_df)}")
print(missing_df["day_of_week"].value_counts())

In [0]:
print(missing_df.head(20))

In [0]:
zero_sales_rows = silver[silver["units_sold"] == 0]
print(f"Rows with units_sold == 0: {len(zero_sales_rows)}")
zero_sales_rows.head(10)

In [0]:
from src.transform.build_gold_features import build_gold_overall

gold_overall = build_gold_overall(silver)
print(gold_overall.shape)
gold_overall.head(15)

In [0]:
# Confirm every calendar day is now present, no gaps
print("Total rows:", len(gold_overall))
print("Date range:", gold_overall["posting_date"].min(), "to", gold_overall["posting_date"].max())

expected_days = (gold_overall["posting_date"].max() - gold_overall["posting_date"].min()).days + 1
print("Expected days:", expected_days, "| Actual rows:", len(gold_overall))

# Confirm the filled-zero days match what we found earlier (~407)
print("Filled (previously missing) days:", gold_overall["was_filled"].sum())

# Check lag/rolling features look sane
print(gold_overall[["lag_7", "rolling_avg_7"]].describe())

# Look at overall sales trend
print(gold_overall["total_units_sold"].describe())

In [0]:
# Find the biggest sales days
print(gold_overall.nlargest(10, "total_units_sold")[["posting_date", "total_units_sold", "day_of_week", "month"]])

In [0]:
display(gold_overall[["posting_date", "total_units_sold"]])

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 5))
plt.plot(gold_overall["posting_date"], gold_overall["total_units_sold"])
plt.title("Daily Battery Sales (EXIDE) — Full History")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(16, 5))
plt.plot(gold_overall["posting_date"], gold_overall["total_units_sold"], alpha=0.3, label="Daily")
plt.plot(gold_overall["posting_date"], gold_overall["rolling_avg_7"], color="red", label="7-day rolling avg")
plt.legend()
plt.title("Daily Sales vs 7-Day Rolling Average")
plt.tight_layout()
plt.show()

In [0]:
gold_overall["day_of_month"] = gold_overall["posting_date"].dt.day
avg_by_day_of_month = gold_overall.groupby("day_of_month")["total_units_sold"].mean()

import matplotlib.pyplot as plt
plt.figure(figsize=(12, 4))
avg_by_day_of_month.plot(kind="bar")
plt.title("Average Units Sold by Day of Month")
plt.xlabel("Day of Month")
plt.ylabel("Average Units Sold")
plt.tight_layout()
plt.show()

In [0]:
print(gold_overall[["posting_date", "day_of_month", "days_until_month_end", "is_month_end", "total_units_sold"]].tail(30))

In [0]:
# Check the most recent 30 days of raw silver data (not aggregated) - are there simply very few rows?
recent_silver = silver[silver["posting_date"] >= "2026-06-01"]
print(recent_silver.groupby(silver["posting_date"].dt.date).size())

In [0]:
# Check daily row counts across the last 90 days, to see if this is an isolated recent issue
recent_90 = silver[silver["posting_date"] >= silver["posting_date"].max() - pd.Timedelta(days=90)]
daily_counts_90 = recent_90.groupby(recent_90["posting_date"].dt.date).size()
print(daily_counts_90.describe())
print(daily_counts_90.tail(30))

In [0]:
cutoff_date = pd.Timestamp("2026-06-12")
gold_overall_trimmed = gold_overall[gold_overall["posting_date"] <= cutoff_date].copy()

print(f"Original rows: {len(gold_overall)}, Trimmed rows: {len(gold_overall_trimmed)}")
print(gold_overall_trimmed.tail(10))

In [0]:
print(f"Shape: {gold_overall_trimmed.shape}")
print(f"Columns: {gold_overall_trimmed.columns.tolist()}")
gold_overall_trimmed.head(10)

In [0]:
gold_overall_trimmed.tail(10)

In [0]:
pd.set_option("display.max_columns", None)
print(gold_overall_trimmed.head(10))
print(gold_overall_trimmed.tail(10))

In [0]:
import io

buffer = io.BytesIO()
gold_overall_trimmed.to_parquet(buffer, index=False)
buffer.seek(0)

blob_client = blob_service.get_blob_client(
    container="gold",
    blob="erp/battery/phase1_overall_daily.parquet"
)
blob_client.upload_blob(buffer, overwrite=True)

print(f"Saved {len(gold_overall_trimmed)} rows to gold/erp/battery/phase1_overall_daily.parquet")

In [0]:
container_client = blob_service.get_container_client("gold")
for blob in container_client.list_blobs(name_starts_with="erp/battery/"):
    print(blob.name, blob.size)

In [0]:
feature_cols = [
    "day_of_week", "month", "is_weekend",
    "day_of_month", "days_until_month_end", "is_month_end",
    "lag_7", "rolling_avg_7", "rolling_avg_30", "prev_month_peak"
]
target_col = "total_units_sold"

# Drop rows without enough history for lag/rolling features to be valid
model_data = gold_overall_trimmed.dropna(subset=feature_cols + [target_col]).copy()

print(f"Rows available for training: {len(model_data)}")

# Chronological split — last 20% of the timeline becomes the test set
# (never shuffle time series data — that would leak future info into training)
split_idx = int(len(model_data) * 0.8)
train_data = model_data.iloc[:split_idx]
test_data = model_data.iloc[split_idx:]

print(f"Train: {len(train_data)} rows, {train_data['posting_date'].min()} to {train_data['posting_date'].max()}")
print(f"Test: {len(test_data)} rows, {test_data['posting_date'].min()} to {test_data['posting_date'].max()}")

X_train, y_train = train_data[feature_cols], train_data[target_col]
X_test, y_test = test_data[feature_cols], test_data[target_col]

In [0]:
#%pip install lightgbm

In [0]:
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

with mlflow.start_run(run_name="phase1_overall_baseline_v2"):
    params = {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 6,
    }
    mlflow.log_params(params)

    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    wape_score = wape(y_test, preds)

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("wape", wape_score)
    mlflow.lightgbm.log_model(model, name="model")

    print(f"Model   — MAE: {mae:.2f}   RMSE: {rmse:.2f}   WAPE: {wape_score:.3%}")

    baseline_preds = X_test["rolling_avg_7"]
    baseline_mae = mean_absolute_error(y_test, baseline_preds)
    baseline_rmse = mean_squared_error(y_test, baseline_preds) ** 0.5
    baseline_wape = wape(y_test, baseline_preds)

    mlflow.log_metric("baseline_mae", baseline_mae)
    mlflow.log_metric("baseline_rmse", baseline_rmse)
    mlflow.log_metric("baseline_wape", baseline_wape)

    print(f"Baseline — MAE: {baseline_mae:.2f}   RMSE: {baseline_rmse:.2f}   WAPE: {baseline_wape:.3%}")

In [0]:
results = test_data.copy()
results["prediction"] = preds

# Split into month-end vs regular days
month_end_days = results[results["days_until_month_end"] <= 2]
regular_days = results[results["days_until_month_end"] > 2]

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

print(f"Regular days WAPE: {wape(regular_days['total_units_sold'], regular_days['prediction']):.3%}  ({len(regular_days)} days)")
print(f"Month-end days WAPE: {wape(month_end_days['total_units_sold'], month_end_days['prediction']):.3%}  ({len(month_end_days)} days)")

In [0]:
from sklearn.metrics import mean_absolute_error

print(f"Regular days — MAE: {mean_absolute_error(regular_days['total_units_sold'], regular_days['prediction']):.2f}  (avg actual: {regular_days['total_units_sold'].mean():.2f})")
print(f"Month-end days — MAE: {mean_absolute_error(month_end_days['total_units_sold'], month_end_days['prediction']):.2f}  (avg actual: {month_end_days['total_units_sold'].mean():.2f})")